# Hybrid RAG for ERP — notebook roboczy

Sekcje 0–1 raz. Sekcje 2–7 przy każdej zmianie korpusu.

**Po zmianie `.env` restartuj kernel** — `autoreload` nie odświeża wartości czytanych przy imporcie.


In [ ]:
%load_ext autoreload
%autoreload 2


---
## 0. Instalacja, modele, baza


In [ ]:
!pip install -r requirements.txt
!pip install fastapi uvicorn requests


In [ ]:
from app.core import LLM_MODEL, EMBED_MODEL, GRAPH_MODEL

print(f'{LLM_MODEL=}\n{EMBED_MODEL=}\n{GRAPH_MODEL=}')


In [ ]:
!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}
!ollama pull {GRAPH_MODEL}


In [ ]:
!docker compose -f database/docker-compose.yml --env-file .env up -d


---
## 1. Sanity check

Jeden import wyłapie literówki, braki w `.env` i złe ścieżki.


In [1]:
import app.core, app.graph, app.schema, app.ingest, app.plan, app.assistant

from app.graph import PROMPTS_DIR, GRAPHS_DIR
from app.ingest import KNOWLEDGE_DIR

print('importy OK')
print(f'{PROMPTS_DIR}  {PROMPTS_DIR.exists()}')
print(f'{KNOWLEDGE_DIR}  {KNOWLEDGE_DIR.exists()}')


importy OK
C:\Users\BartoszZieliński\Desktop\Asystent-erp\Hybrid-RAG-For-ERP-Management\system  True
C:\Users\BartoszZieliński\Desktop\Asystent-erp\Hybrid-RAG-For-ERP-Management\knowledge  True


### Czy schema.py ma nowe pola


In [2]:
from app.schema import Procedure, ProcedureStep

for pole in ('requires', 'provides'):
    assert pole in ProcedureStep.model_fields, f'ProcedureStep bez pola {pole}'
assert 'goal' in Procedure.model_fields, 'Procedure bez pola goal'
print('schema.py OK')


schema.py OK


### Czy korpus jest poprawny jeszcze przed ingestem


In [3]:
from app.ingest import load_knowledge, check_for_duplicates
from app.schema import Procedure

docs = load_knowledge()
check_for_duplicates(docs)

procedury = [d for d in docs if isinstance(d, Procedure)]
kroki = sum(len(p.steps) for p in procedury)
bez_celu = [p.id for p in procedury if not p.goal]
bez_stanow = [p.id for p in procedury if not any(s.provides for s in p.steps)]

print(f'Dokumentów: {len(docs)}, procedur: {len(procedury)}, kroków: {kroki}')
print('Bez goal:', bez_celu or 'brak')
print('Bez provides:', bez_stanow or 'brak')


Dokumentów: 78, procedur: 40, kroków: 177
Bez goal: brak
Bez provides: brak


### Symulacja korpusu offline

Sprawdza, czy w obrębie procedury każdy warunek jest wcześniej wytworzony.
Braki na starcie procedury są **normalne** — to zależności międzyproceduralne.


In [4]:
daje_globalnie = {s for p in procedury for st in p.steps for s in st.provides}

for p in procedury:
    stan, lokalne = set(), []
    for i, st in enumerate(p.steps, 1):
        brak = set(st.requires) - stan
        if brak: lokalne.append((i, sorted(brak)))
        stan |= set(st.provides)
    brak_celu = set(p.goal) - stan
    if lokalne or brak_celu:
        print(f'{p.id}')
        for i, b in lokalne: print(f'   krok {i}: {b}')
        if brak_celu: print(f'   CEL nieosiągnięty: {sorted(brak_celu)}')

wymagane = {s for p in procedury for st in p.steps for s in st.requires}
print('\nStany bez producenta (BŁĄD, jeśli niepuste):', sorted(wymagane - daje_globalnie) or 'brak')
print('Stanów łącznie:', len(daje_globalnie))


proc.inwentaryzacja.wprowadzenie-liczen
   krok 1: ['inw.arkusz-otwarty']
proc.inwentaryzacja.zamkniecie
   krok 1: ['inw.arkusz-otwarty']
   krok 2: ['inw.policzone']
proc.magazyn.zapis-szkicu
   krok 1: ['dokument.nowy']
proc.magazyn.numer-obcy
   krok 1: ['dokument.nowy']
proc.magazyn.usuniecie-pozycji
   krok 1: ['dokument.nowy']
proc.magazyn.lokalizacja-na-pozycji
   krok 1: ['dokument.nowy']
   krok 2: ['pozycja.produkt']
proc.magazyn.zmiana-ceny-pozycji
   krok 1: ['dokument.nowy']
   krok 2: ['pozycja.produkt']
proc.magazyn.zatwierdzenie
   krok 1: ['dokument.nowy']
   krok 3: ['pozycja.ilosc']
proc.sprzedaz.potwierdzenie-zamowienia
   krok 1: ['zs.zapisane']
proc.sprzedaz.realizacja-zamowienia
   krok 1: ['zs.potwierdzone']
proc.sprzedaz.podglad-wz-z-zamowienia
   krok 1: ['zs.zrealizowane']
proc.zakupy.wyslanie-zamowienia
   krok 1: ['zz.zapisane']
proc.zakupy.przyjecie-dostawy
   krok 1: ['zz.wyslane']
proc.zakupy.rejestracja-faktury
   krok 3: ['zz.wyslane']
proc.zakupy.poz

---
## 2. Ingest

`purge_database` jest obowiązkowy — hash kroku zmienił się wraz ze stanami.


In [1]:
from app import graph

graph.initialize_graph_driver()
graph.purge_database(driver=graph.graph_driver)
print('baza wyczyszczona')


OK: Połączenie z neo4j działa
APOC jest dostępne.
baza wyczyszczona


Zatrzyma się na `input()`. Sprawdź linię **`Dodano N kroków...`** zanim naciśniesz ENTER.


In [2]:
from app import core
from app import graph
from app.ingest import ingest_llm

ingest_llm(driver=graph.graph_driver, model=core.GRAPH_MODEL)


1
2
3
4
5
{'id': 'proc.inwentaryzacja.rozpoczecie', 'title': 'Rozpoczęcie inwentaryzacji magazynu', 'module': 'inwentaryzacja', 'summary': 'Jak otworzyć arkusz spisu z natury dla wybranego magazynu i zamrozić stan księgowy.', 'query': ['jak zacząć inwentaryzację', 'jak zrobić spis z natury', 'nowy arkusz inwentaryzacji', 'jak policzyć magazyn', 'remanent jak rozpocząć'], 'preconditions': ['Dla wybranego magazynu nie może istnieć inna otwarta inwentaryzacja'], 'roles': ['magazynier', 'kierownik'], 'steps': [{'text': 'Przejdź do Inwentaryzacja w menu bocznym', 'anchor': 'nav.stocktakes', 'action': {'kind': 'click'}, 'optional': False, 'requires': [], 'provides': ['ekran.inwentaryzacja']}, {'text': 'Wybierz magazyn do policzenia z listy obok przycisku', 'anchor': 'field.stocktake-warehouse', 'action': {'kind': 'select', 'label': 'MAG-GL'}, 'optional': False, 'requires': ['ekran.inwentaryzacja'], 'provides': ['inw.magazyn-wybrany']}, {'text': 'Kliknij Nowa inwentaryzacja', 'anchor': 'btn.s

Zapisywanie relacji: 100%|██████████| 584/584 [00:00<00:00, 1263.31it/s]

{'nodes': 350, 'relations': 584, 'embeddings': 219}

Gotowe!
Możesz podglądać wyniki na: http://localhost:7474


---
## 3. Weryfikacja grafu


In [3]:
kg = graph.knowledge_graph

print('Klasy:  ', list(kg.classes.keys()))
print('Relacje:', list(kg.relations.keys()))

from collections import Counter
print('\nWęzły wg klas:', Counter(n.c_name for n in kg.nodes.values()))
print('Węzły wg modułów:', Counter(n.module for n in kg.nodes.values()))


Klasy:   ['Procedura', 'Blad', 'Pojecie', 'Krok', 'Stan']
Relacje: ['DOTYCZY', 'ROZWIAZYWANY_PRZEZ', 'WYMAGA', 'POWIAZANE_Z', 'MA_KROK', 'WYMAGA_STANU', 'DAJE_STAN', 'MA_CEL']

Węzły wg klas: Counter({'Krok': 141, 'Stan': 131, 'Procedura': 40, 'Blad': 23, 'Pojecie': 15})
Węzły wg modułów: Counter({'magazyn': 122, 'zakupy': 87, 'sprzedaz': 80, 'lokalizacje': 32, 'inwentaryzacja': 29})


### Zgodność nazw węzłów z konwencją


In [4]:
from app.plan import node_id_from_document_id

for p in procedury:
    ocz = node_id_from_document_id(p.id)
    print(f"  {p.id:45} {'OK' if ocz in kg.nodes else 'BRAK W GRAFIE'}")


NameError: name 'procedury' is not defined

### Walidacja korpusu w bazie

`stany_nieuzywane` wskaże stany końcowe (np. `inw.zamknieta`) — to nie błąd.
Pozostałe kategorie muszą być puste.


In [5]:
from app.plan import validate_corpus

problemy = validate_corpus(graph.graph_driver)

for kategoria, lista in problemy.items():
    print(f'\n{kategoria}  ({len(lista)})')
    for x in lista[:15]: print('   ', x)
if not problemy: print('Korpus spójny')



stany_nieuzywane  (50)
    inw.roznice-sprawdzone
    inw.zamknieta
    inw.postep-sprawdzony
    lok.pojemnosc
    lok.opis
    lok.zapisana
    lok.dezaktywowana
    lok.filtr-magazyn
    lok.aktywowana
    dokument.typ-pz
    dokument.kontrahent
    dokument.magazyn-docelowy
    dokument.zatwierdzony
    dokument.typ-wz
    dokument.typ-mm


---
## 4. Testy planowania

Tu sprawdzasz sedno: czy trawersja daje sensowne, kompletne plany.


In [6]:
from app.plan import build_plan, full_plan, plan_for_goal, load_step_index, goal_states

index = load_step_index(graph.graph_driver)
print('Kroków w indeksie:', len(index))


Kroków w indeksie: 141


**Plan podstawowy** — kolejność redakcyjna z YAML-a:


In [7]:
for r in build_plan(graph.graph_driver, 'proc_magazyn_przyjecie_pz'):
    print(f"  {r['tekst'][:60]}")


  Przejdź do Dokumenty w menu bocznym
  Kliknij Nowy dokument
  W polu Typ dokumentu wybierz PZ
  W polu Dostawca wybierz kontrahenta, od którego przyjmujesz 
  W sekcji Magazyny wybierz Magazyn docelowy
  Przejdź na zakładkę Pozycje
  Kliknij Dodaj pozycję
  Wybierz produkt z listy
  Wpisz ilość w pole pozycji
  Kliknij Zatwierdź dokument


**Łańcuch międzyproceduralny** — realizacja zamówienia sprzedaży wymaga
utworzenia i potwierdzenia zamówienia. `full_plan` dokleja je jako prefiks.


In [8]:
rows = full_plan(graph.graph_driver, 'proc_sprzedaz_realizacja_zamowienia', index=index)

for i, r in enumerate(rows, 1):
    print(f"  {i:2}. [{r['procedura'][:22]:22}] {r['tekst'][:52]}")


   1. [(warunek wstępny)     ] Przejdź do Zamówienia sprzedaży w menu bocznym
   2. [(warunek wstępny)     ] Kliknij Nowe zamówienie
   3. [(warunek wstępny)     ] Ustaw oczekiwaną datę realizacji
   4. [(warunek wstępny)     ] W polu Magazyn wydania wybierz, z którego magazynu z
   5. [(warunek wstępny)     ] W polu Odbiorca wybierz klienta, który składa zamówi
   6. [(warunek wstępny)     ] W stopce karty Pozycje kliknij Dodaj pozycję
   7. [(warunek wstępny)     ] Wybierz produkt z listy
   8. [(warunek wstępny)     ] Wpisz zamawianą ilość
   9. [(warunek wstępny)     ] Kliknij Zapisz zamówienie w stopce karty
  10. [(warunek wstępny)     ] Otwórz szkic zamówienia klikając jego numer na liści
  11. [(warunek wstępny)     ] W stopce karty Pozycje kliknij Potwierdź zamówienie
  12. [proc_sprzedaz_realizac] Otwórz zamówienie w statusie Potwierdzone klikając j
  13. [proc_sprzedaz_realizac] W stopce karty Pozycje kliknij Zrealizuj (utwórz WZ)
  14. [proc_sprzedaz_realizac] W szkicu WZ z

**Ten sam cel z kontekstem** — użytkownik ma już potwierdzone zamówienie:


In [9]:
rows = full_plan(graph.graph_driver, 'proc_sprzedaz_realizacja_zamowienia',
                 stan_poczatkowy={'zs.potwierdzone'}, index=index)

print(f'{len(rows)} kroków:')
for r in rows: print(f"   {r['tekst'][:60]}")


4 kroków:
   Przejdź do Zamówienia sprzedaży w menu bocznym
   Otwórz zamówienie w statusie Potwierdzone klikając jego nume
   W stopce karty Pozycje kliknij Zrealizuj (utwórz WZ)
   W szkicu WZ zweryfikuj ilości i zatwierdź dokument


**Planowanie czyste od celu** — bez redakcji, z wyborem najtańszej ścieżki:


In [10]:
for cel in (['dokument.typ-pz', 'dokument.zatwierdzony'], ['fz.zaksiegowana'], ['inw.zamknieta']):
    p = plan_for_goal(cel, index)
    moduly = sorted({k['modul'] for k in p})
    print(f'{str(cel):55} -> {len(p):2} kroków, moduły: {moduly}')


['dokument.typ-pz', 'dokument.zatwierdzony']            ->  7 kroków, moduły: ['inwentaryzacja', 'magazyn']
['fz.zaksiegowana']                                     ->  8 kroków, moduły: ['zakupy']
['inw.zamknieta']                                       ->  8 kroków, moduły: ['inwentaryzacja']


**Test negatywny** — odwrócony plan MUSI zgłosić błędy:


In [11]:
from app.plan import validate_plan

kroki = [r['krok_id'] for r in build_plan(graph.graph_driver, 'proc_magazyn_przyjecie_pz')]

print('Poprawny :', validate_plan(graph.graph_driver, kroki) or 'bez zarzutu')
print('Odwrócony:', validate_plan(graph.graph_driver, list(reversed(kroki)))[:3])


Poprawny : bez zarzutu
Odwrócony: ["Krok 1 (krok_0abcfe20f8ad) wymaga niespełnionych stanów: ['pozycja.ilosc']", "Krok 2 (krok_9777f803763a) wymaga niespełnionych stanów: ['pozycja.produkt']", "Krok 3 (krok_5211aeb1612a) wymaga niespełnionych stanów: ['pozycja.nowa']"]


---
## 5. Strojenie MIN_SCORE

Ustaw `ASSISTANT_MIN_SCORE` **powyżej** najlepszego trafienia dla pytań spoza korpusu,
**poniżej** najgorszego dla pytań sensownych. Przy 5 modułach to ważniejsze niż wcześniej.


In [12]:
graph.initialize_embed_model()

pytania = [
    ('W korpusie', 'jak przyjąć towar na magazyn'),
    ('W korpusie', 'jak wysłać zamówienie do dostawcy'),
    ('W korpusie', 'jak zamknąć inwentaryzację'),
    ('W korpusie', 'jak dodać nowy regał'),
    ('W korpusie', 'jak zarejestrować fakturę od dostawcy'),
    ('SPOZA',      'jaka jest stolica Francji'),
    ('SPOZA',      'jak ugotować makaron'),
]

for etykieta, q in pytania:
    w = graph.embed_model.encode(q)[0]
    wyniki = graph.KnowledgeGraph.search_semantic(graph.graph_driver, w, top_k=3)
    print(f'\n[{etykieta}] {q}')
    for r in wyniki:
        print(f"   {r['score']:.3f}  {r.get('klasa',''):10} {r.get('modul',''):15} {r['node_id']}")



[W korpusie] jak przyjąć towar na magazyn
   0.790  Krok                       krok_6e647bbc5b9b
   0.776  Krok                       krok_06b069d07673
   0.775  Krok                       krok_3bb3adfd2371

[W korpusie] jak wysłać zamówienie do dostawcy
   0.823  Krok                       krok_043e8efeb54d
   0.807  Krok                       krok_f283f0dce796
   0.807  Krok                       krok_1d6741cd573b

[W korpusie] jak zamknąć inwentaryzację
   0.853  Krok                       krok_ba777462b57c
   0.838  Procedura                  proc_inwentaryzacja_zamkniecie
   0.810  Blad                       ERR_4003

[W korpusie] jak dodać nowy regał
   0.778  Krok                       krok_5ad92026456d
   0.778  Krok                       krok_713e1dd4cd98
   0.771  Krok                       krok_a64981a9da61

[W korpusie] jak zarejestrować fakturę od dostawcy
   0.861  Krok                       krok_43f8eae5b2e6
   0.852  Krok                       krok_9a607d707c43
   0.84

### Filtr modułu


In [13]:
w = graph.embed_model.encode('jak zamówić towar')[0]

for m in [None, 'zakupy', 'magazyn', 'lokalizacje']:
    r = graph.KnowledgeGraph.search_semantic(graph.graph_driver, w, top_k=5, module=m)
    print(f'module={str(m):14} -> {len(r)} wyników, top: {r[0]["node_id"] if r else "-"}')


module=None           -> 5 wyników, top: krok_c8ba76d11f1f
module=zakupy         -> 5 wyników, top: krok_c8ba76d11f1f
module=magazyn        -> 1 wyników, top: krok_ab9eb02bcc00
module=lokalizacje    -> 0 wyników, top: -


---
## 6. Warstwa asystenta (bez HTTP)

Łatwiej debugować niż przez serwer.


In [14]:
from app.assistant import answer, get_index

get_index(reload=True)

def zapytaj(q, ctx=None):
    o = answer(q, ctx)
    print(f'\n=== {q}   {ctx or ""}')
    print(f"   refused={o['refused']}  kroków={len(o['steps'])}  sources={o['sources']}")
    print(f"   {o['text'][:110]}")
    for s in o['steps'][:4]:
        print(f"      - {s['text'][:55]}  action={s.get('action', {}).get('kind')}")
    return o


### Zestaw testowy — pokrywa wszystkie ścieżki


In [15]:
# procedura w jednym module
zapytaj('jak przyjąć towar na magazyn')
zapytaj('jak dodać nową lokalizację')

# łańcuch międzyproceduralny
zapytaj('jak wysłać towar do klienta z zamówienia')
zapytaj('jak zamknąć inwentaryzację')



=== jak przyjąć towar na magazyn   
   refused=False  kroków=10  sources=['proc_magazyn_przyjecie_pz']
   Aby przyjąć towar na magazyn, zatwierdź dokument, co zwiększy stan produktu.
      - Przejdź do Dokumenty w menu bocznym  action=click
      - Kliknij Nowy dokument  action=click
      - W polu Typ dokumentu wybierz PZ  action=select
      - W polu Dostawca wybierz kontrahenta, od którego przyjmu  action=select

=== jak dodać nową lokalizację   
   refused=False  kroków=7  sources=['proc_lokalizacje_dodanie_lokalizacji']
   Nowa lokalizacja zostaje dodana do listy ze statusem „Aktywna”.
      - Przejdź do Lokalizacje w menu bocznym  action=click
      - Kliknij Nowa lokalizacja  action=click
      - W polu Magazyn wybierz magazyn, w którym powstaje lokal  action=select
      - Wpisz kod lokalizacji w układzie regał-poziom  action=fill

=== jak wysłać towar do klienta z zamówienia   
   refused=True  kroków=0  sources=[]
   Nie znalazłem tego w bazie wiedzy. Mogę natomiast pomóc z 

{'text': 'Aby zamknąć inwentaryzację, zmień status arkusza na „Zamknięta”, co ograniczy go do odczytu.',
 'steps': [{'text': 'Przejdź do Inwentaryzacja w menu bocznym',
   'anchor': 'nav.stocktakes',
   'action': {'kind': 'click', 'anchor': 'nav.stocktakes'}},
  {'text': 'Wybierz magazyn do policzenia z listy obok przycisku',
   'anchor': 'field.stocktake-warehouse',
   'action': {'kind': 'select',
    'anchor': 'field.stocktake-warehouse',
    'label': 'MAG-GL'}},
  {'text': 'Kliknij Nowa inwentaryzacja',
   'anchor': 'btn.stocktake-new',
   'action': {'kind': 'click', 'anchor': 'btn.stocktake-new'},
   'note': 'System zapisuje stan księgowy z tej chwili jako punkt odniesienia — dokumenty zatwierdzane w trakcie liczenia nie zmienią arkusza'},
  {'text': 'Otwórz arkusz klikając jego numer na liście inwentaryzacji',
   'anchor': 'table.stocktakes',
   'note': 'Otwarte arkusze mają status Otwarta'},
  {'text': 'Dla każdej pozycji wpisz ilość policzoną fizycznie w kolumnie Policzono',
   

In [ ]:
# kontekst skraca plan
zapytaj('jak zrealizować zamówienie sprzedaży')
zapytaj('jak zrealizować zamówienie sprzedaży', {'salesOrderStatus': 'confirmed'})


In [16]:
# zawężenie modułem, pojęcie, błąd, odmowa
zapytaj('jak zamówić towar', {'module': 'zakupy'})
zapytaj('co to jest dokument PZ')
zapytaj('nie mogę zapisać dokumentu', {'module': 'magazyn', 'lastError': {'code': 'ERR-1004'}})
zapytaj('jaka jest stolica Francji')



=== jak zamówić towar   {'module': 'zakupy'}
   refused=False  kroków=11  sources=['proc_zakupy_utworzenie_zamowienia']
   Utwórz zamówienie, aby otrzymało numer ZZ i status „Szkic”, a następnie otwórz jego podgląd.
      - Przejdź do Zamówienia zakupu w menu bocznym  action=click
      - Kliknij Nowe zamówienie  action=click
      - W polu Dostawca wybierz kontrahenta, u którego zamawias  action=select
      - W polu Magazyn dostawy wybierz, dokąd ma trafić towar  action=select

=== co to jest dokument PZ   
   refused=False  kroków=10  sources=['proc_magazyn_przyjecie_pz', 'concept_dokument_magazynowy']
   Dokument PZ (Przyjęcie Zewnętrzne) potwierdza przyjęcie towarów do magazynu od zewnętrznego dostawcy.
      - Przejdź do Dokumenty w menu bocznym  action=click
      - Kliknij Nowy dokument  action=click
      - W polu Typ dokumentu wybierz PZ  action=select
      - W polu Dostawca wybierz kontrahenta, od którego przyjmu  action=select

=== nie mogę zapisać dokumentu   {'module': 

{'text': 'Nie znalazłem tego w bazie wiedzy. Mogę natomiast pomóc z tym: ERR-6002.',
 'steps': [],
 'sources': [],
 'refused': True}

---
## 7. API

Serwer w **osobnym terminalu**:

```bash
uvicorn app.api:app --reload --port 8000
```


In [19]:
import requests
print(requests.get('http://localhost:8000/assistant/health').json())


{'ok': True, 'wezly': 350, 'z_embeddingami': 219}


### Asercja kontraktu AssistantReply


In [20]:
import requests

def sprawdz(q, ctx=None):
    o = requests.post('http://localhost:8000/assistant/ask',
                      json={'question': q, 'context': ctx}).json()
    assert set(o) == {'text','steps','sources','refused'}, o.keys()
    assert isinstance(o['text'], str) and isinstance(o['refused'], bool)
    assert isinstance(o['sources'], list)
    for s in o['steps']:
        assert set(s) <= {'text','anchor','action','note'}, s
        assert isinstance(s['text'], str) and s['text']
        if 'action' in s:
            assert s['action']['kind'] in {'navigate','click','fill','select'}, s['action']
            assert s['action'].get('anchor') or s['action']['kind'] == 'navigate', s['action']
    print(f"OK  refused={o['refused']}  kroków={len(o['steps'])}  {q}")
    return o

sprawdz('jak przyjąć towar na magazyn')
sprawdz('jak wysłać towar do klienta z zamówienia')
sprawdz('co to jest dokument PZ')
sprawdz('jaka jest stolica Francji')
sprawdz('jak zrealizować zamówienie', {'salesOrderStatus': 'confirmed'})


OK  refused=False  kroków=10  jak przyjąć towar na magazyn
OK  refused=False  kroków=17  jak wysłać towar do klienta z zamówienia
OK  refused=False  kroków=16  co to jest dokument PZ
OK  refused=True  kroków=0  jaka jest stolica Francji
OK  refused=False  kroków=17  jak zrealizować zamówienie


{'text': 'Aby zrealizować zamówienie, ustaw jego status na „Zrealizowane" i zatwierdź wydanie wewnętrzne.',
 'steps': [{'text': 'Przejdź do Zamówienia sprzedaży w menu bocznym',
   'anchor': 'nav.sales-orders',
   'action': {'kind': 'click', 'anchor': 'nav.sales-orders'}},
  {'text': 'Kliknij Nowe zamówienie',
   'anchor': 'btn.so-new',
   'action': {'kind': 'click', 'anchor': 'btn.so-new'}},
  {'text': 'W polu Odbiorca wybierz klienta, który składa zamówienie',
   'anchor': 'field.so-customer',
   'action': {'kind': 'select',
    'anchor': 'field.so-customer',
    'label': 'Metalpol'},
   'note': 'Lista pokazuje tylko odbiorców — dostawcy są ukryci'},
  {'text': 'W polu Magazyn wydania wybierz, z którego magazynu zejdzie towar',
   'anchor': 'field.so-warehouse',
   'action': {'kind': 'select',
    'anchor': 'field.so-warehouse',
    'label': 'MAG-GL'}},
  {'text': 'Ustaw oczekiwaną datę realizacji',
   'anchor': 'field.so-expected-date',
   'action': {'kind': 'fill',
    'anchor': 'f

### Podgląd pełnej odpowiedzi z akcjami autopilota


In [ ]:
o = sprawdz('jak przyjąć towar na magazyn')

print('\n' + o['text'] + '\n')
for i, s in enumerate(o['steps'], 1):
    print(f"{i}. {s['text']}")
    print(f"      anchor={s.get('anchor')}  action={s.get('action')}")
    if s.get('note'): print(f"      uwaga: {s['note']}")


### Zbieranie realnego `context` z frontu

Dodaj w `api.py` logowanie `req.context`, poklikaj widget na różnych ekranach
i zbierz próbki — na ich podstawie uzupełnij `STATE_FROM_CONTEXT` w `assistant.py`.


In [ ]:
# po ponownym ingeście przeładuj bufor indeksu w działającym serwerze
print(requests.post('http://localhost:8000/assistant/reload').json())


## 8. Kopie zapasowe grafu


In [ ]:
from app.graph import GRAPHS_DIR

for f in sorted(GRAPHS_DIR.glob('*.json'), reverse=True)[:10]:
    print(f'{f.stat().st_size/1024:8.1f} KB  {f.name}')


In [ ]:
# graph.load_graph(GRAPHS_DIR / 'NAZWA.json')
# print(len(graph.knowledge_graph.nodes), 'węzłów')


In [17]:
graph.save_graph("gpt-5.6-luna", graph.knowledge_graph, with_embeddings=True, embed_model="bge-m3")

WindowsPath('C:/Users/BartoszZieliński/Desktop/Asystent-erp/Hybrid-RAG-For-ERP-Management/knowledge/graphs/2026-08-10_14-26-11_gpt-5.6-luna.json')